# Identifying topic using zero-shot modelling 

Going to generate some intial labels for my training dataset using zero shot modelling. 

In [1]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict
import string 
from datasets import Dataset
import regex

from transformers import AutoModelForMaskedLM, pipeline, AutoTokenizer, AutoModel 
import torch
from sentence_transformers import SentenceTransformer

from torch.nn.functional import cosine_similarity

from sklearn.model_selection import GroupShuffleSplit

import sys
sys.path.append('../pipeline')
from nlp_tasks import NLP_Tasks
from comments_saver import CommentsSaver

/opt/miniconda3/envs/nlp_env_analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cs = CommentsSaver(env='dev')
df = cs.read_all()

Connecting to the ai4ci-db-dev database...
Successfully connected to ai4ci-db-dev.


In [3]:
df.head()

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon
0,70254,Westminster,24/01842/FULL_2,24/01842/FULL,None,Objects,2024-05-08,I am the owner of 9 Montagu Mews North and wis...,2025-04-04,NaN,NaN
1,70255,Westminster,24/01842/FULL_3,24/01842/FULL,None,Objects,2024-05-07,We greatly value the olive tree (15 cm in diam...,2025-04-04,NaN,NaN
2,70256,Barnet,23/4878/FUL_1,23/4878/FUL,34 The Ridgeway London NW11 8QS,Objects,2023-12-14,The application is not in keeping with the are...,2025-04-04,NaN,NaN
3,70257,Barnet,23/4878/FUL_2,23/4878/FUL,40 The Ridgeway LONDON NW11 8QS,Objects,2023-12-14,I strongly object to the above planning applic...,2025-04-04,NaN,NaN
4,70258,Barnet,23/4878/FUL_3,23/4878/FUL,43A Woodstock Road London NW11 8ES,Objects,2023-12-11,I object. Overdevelopment of a currently peace...,2025-04-04,NaN,NaN


In [4]:
# Assume df is your DataFrame and 'application_id' is the column to group by
gss = GroupShuffleSplit(n_splits=1, train_size=0.05, random_state=42)

# Get the application_id groups
groups = df['application_id']

# Split the indices
train_inds, test_inds = next(gss.split(df, groups=groups))

# Create train and test DataFrames
df_train = df.iloc[train_inds].reset_index(drop=True)
#df_test = df.iloc[test_inds].reset_index(drop=True)

In [5]:
len(df_train)

1379

In [6]:
df_train.head()

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon
0,70367,Ealing,221960FUL_1,221960FUL,65 St. Mary's Road Ealing W5 5RG W5 5RG,Objects,2022-06-17,We strongly object to the proposed development...,2025-04-04,NaN,NaN
1,70368,Ealing,221960FUL_2,221960FUL,Ealing Green and Town Centre Conservation Pane...,Objects,2022-06-14,The Ealing Green and Ealing Town Centre Conser...,2025-04-04,NaN,NaN
2,70443,Barnet,22/5566/FUL_1,22/5566/FUL,36 Hermitage Lane London NW2 2HD,Objects,2023-04-17,On the 11th April 2023 I received a letter fro...,2025-04-04,NaN,NaN
3,70444,Barnet,22/5566/FUL_2,22/5566/FUL,36 Hermitage Lane London NW2 2HD,Neutral,2023-04-12,Subsequent to my comments about the original a...,2025-04-04,NaN,NaN
4,70445,Barnet,22/5566/FUL_3,22/5566/FUL,42 Hermitage Lane LONDON NW2 2HD,Neutral,2023-01-08,Commentor: Neighbour\nAddress 42 Hermitage Lan...,2025-04-04,NaN,NaN


In [7]:
df_train['cleaned_comment_text'] = df_train['comment_text'].astype(str)

In [8]:
model_checkpoint = "../outputs/nlp_fine_tuning/distilbert-base-uncased"
nlp = NLP_Tasks(model_checkpoint)

# remove place names, numbers and non-ASCII characters
# df = nlp.remove_place_names(df=df, column='comment_text')
# df = nlp.remove_numbers(df=df, column='cleaned_comment_text')
df_train = nlp.remove_non_ascii(df=df_train, column='cleaned_comment_text')

Device set to use mps:0


In [9]:
df_train = nlp.split_text_on_newline(df=df_train, column='cleaned_comment_text', filter_empty=True, filter_short=True, min_length=5)

In [10]:
len(df_train)

7722

In [11]:
df_train = nlp.split_text_on_period(df=df_train, column='cleaned_comment_text', filter_empty=True, filter_short=True, min_length=5)

In [12]:
len(df_train)

14202

In [13]:
df_train = nlp.remove_empty_rows(df=df_train, column='cleaned_comment_text')

In [14]:
len(df_train)

14189

In [15]:
df_train.head()

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon,cleaned_comment_text
0,70367,Ealing,221960FUL_1,221960FUL,65 St. Mary's Road Ealing W5 5RG W5 5RG,Objects,2022-06-17,We strongly object to the proposed development...,2025-04-04,NaN,NaN,We strongly object to the proposed development
1,70367,Ealing,221960FUL_1,221960FUL,65 St. Mary's Road Ealing W5 5RG W5 5RG,Objects,2022-06-17,We strongly object to the proposed development...,2025-04-04,NaN,NaN,This will put a strain on the existing sewer s...
2,70367,Ealing,221960FUL_1,221960FUL,65 St. Mary's Road Ealing W5 5RG W5 5RG,Objects,2022-06-17,We strongly object to the proposed development...,2025-04-04,NaN,NaN,Mary's Road
3,70367,Ealing,221960FUL_1,221960FUL,65 St. Mary's Road Ealing W5 5RG W5 5RG,Objects,2022-06-17,We strongly object to the proposed development...,2025-04-04,NaN,NaN,We feel it cannot support the addition of anot...
4,70367,Ealing,221960FUL_1,221960FUL,65 St. Mary's Road Ealing W5 5RG W5 5RG,Objects,2022-06-17,We strongly object to the proposed development...,2025-04-04,NaN,NaN,There will also be an increase in noise pollut...


In [17]:
topic_df = pd.read_excel('/Users/bea/Documents/AI4CI/projects/comment_summariser/list_of_topics.xlsx')

In [22]:
topic_list = set(topic_df['Topics'].astype(str).str.lower().tolist())

In [23]:
# strip whitespace
topic_list = [topic.strip() for topic in topic_list]

In [28]:
# remove 'nan' from list
topic_list = [topic for topic in topic_list if topic != 'nan']

In [30]:
topic_list.append('not found')

In [35]:
len(topic_list)

68

In [39]:
topic_list

['housing mix wrong',
 'impact on community',
 'accomodation unsuitable for vulnerable people',
 'loss of view',
 'prior planning decisions',
 'landscaping issues',
 'impact on building infrastructure',
 'solar panels',
 'better replacement accomodation',
 'unaesthetic',
 'increased air pollution',
 'in character with local area',
 'inadequate public consultation',
 'not affordable',
 'scale of development',
 'density of development',
 'loss of privacy',
 'lack of maintenance',
 'archaeology',
 'construction disturbance',
 'height of development',
 'impact on local amenities',
 'drainage and flooding',
 'damage to natural environment',
 'site for sale',
 'hazardous materials',
 'against planning policies',
 'residents forced to leave area',
 'lack of space for social distancing',
 'impact on social infrastructure',
 'will improve safety',
 'applicants character',
 'boundary disputes',
 'impact on transport infrastructure',
 'neighbourhood disputes',
 'volume of objections',
 'increased

## Cosine similarity of embeddings to create intial labels

In [47]:
import torch
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

tqdm.pandas()

# Your model/tokenizer
model_checkpoint = "../outputs/nlp_fine_tuning/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModel.from_pretrained(model_checkpoint)
model.eval()

# Helper function for getting sentence embedding
def get_embedding(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)  # Mean pooling
    return embeddings.squeeze().cpu().numpy()

# Get topic embeddings
topic_embeddings = []
for topic in topic_list:
    combined_text = " ".join(topic) if isinstance(topic, list) else topic
    emb = get_embedding(combined_text, model, tokenizer)
    topic_embeddings.append(emb)

# Convert to tensor matrix for cosine similarity calc
topic_embeddings_matrix = torch.tensor(topic_embeddings)

# Assign topic to each comment
def assign_topic(comment):
    emb = get_embedding(comment, model, tokenizer)
    similarities = cosine_similarity(
        emb.reshape(1, -1),
        topic_embeddings_matrix.numpy()
    )
    max_idx = similarities.argmax()
    # max_score = similarities[0, max_idx]
    return topic_list[max_idx]

# Apply to DataFrame
df_train['topic'] = df_train['cleaned_comment_text'].progress_apply(assign_topic)


100%|██████████| 14189/14189 [08:04<00:00, 29.31it/s]


In [48]:
df_train

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon,cleaned_comment_text,topic
0,70367,Ealing,221960FUL_1,221960FUL,65 St. Mary's Road Ealing W5 5RG W5 5RG,Objects,2022-06-17,We strongly object to the proposed development...,2025-04-04,NaN,NaN,We strongly object to the proposed development,against planning policies
1,70367,Ealing,221960FUL_1,221960FUL,65 St. Mary's Road Ealing W5 5RG W5 5RG,Objects,2022-06-17,We strongly object to the proposed development...,2025-04-04,NaN,NaN,This will put a strain on the existing sewer s...,developer previously built below standard dwel...
2,70367,Ealing,221960FUL_1,221960FUL,65 St. Mary's Road Ealing W5 5RG W5 5RG,Objects,2022-06-17,We strongly object to the proposed development...,2025-04-04,NaN,NaN,Mary's Road,right of way
3,70367,Ealing,221960FUL_1,221960FUL,65 St. Mary's Road Ealing W5 5RG W5 5RG,Objects,2022-06-17,We strongly object to the proposed development...,2025-04-04,NaN,NaN,We feel it cannot support the addition of anot...,developer previously built below standard dwel...
4,70367,Ealing,221960FUL_1,221960FUL,65 St. Mary's Road Ealing W5 5RG W5 5RG,Objects,2022-06-17,We strongly object to the proposed development...,2025-04-04,NaN,NaN,There will also be an increase in noise pollut...,loss of green space
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14197,70142,Southwark,21/AP/3247_33,21/AP/3247,None,Objects,2021-09-27,I have the following objections to this develo...,2025-04-04,NaN,NaN,- Loss of light,loss of light
14198,70142,Southwark,21/AP/3247_33,21/AP/3247,None,Objects,2021-09-27,I have the following objections to this develo...,2025-04-04,NaN,NaN,"Again, when planning permission for (16/AP/523...",developer previously built below standard dwel...
14199,70142,Southwark,21/AP/3247_33,21/AP/3247,None,Objects,2021-09-27,I have the following objections to this develo...,2025-04-04,NaN,NaN,It is hard to believe that additional 9 storey...,impact on building infrastructure
14200,70142,Southwark,21/AP/3247_33,21/AP/3247,None,Objects,2021-09-27,I have the following objections to this develo...,2025-04-04,NaN,NaN,"Sincerely,",not affordable


In [49]:
df_train.to_csv('../outputs/labelled_data/five_percent_labelled_data.csv', index=False)

In [50]:
import random

def generate_topic_metadata(topic_list):
    def random_hex_color():
        return "#{:06x}".format(random.randint(0, 0xFFFFFF))

    result = []
    for topic in topic_list:
        result.append({
            "text": topic,
            "suffix_key": topic[0].lower(),
            "background_color": random_hex_color(),
            "text_color": "#ffffff"
        })
    return result


In [51]:
topic_metadata = generate_topic_metadata(topic_list)

In [53]:
import json
with open("../outputs/labelled_data/topic_metadata.json", "w") as f:
    json.dump(topic_metadata, f, indent=4)

## Zero-shot classification to get us started 

In [ ]:
# classifier = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli")

The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
Device set to use mps:0


In [ ]:
# # Define the candidate labels
# candidate_labels = topic_list

# # Get predictions for each comment
# predictions = []
# for text in cleaned_place_text:
#     result = classifier(text, candidate_labels, multi_label=False)
#     predictions.append(result)
# # Extract the labels and scores
# predicted_labels = []
# predicted_scores = []
# for result in predictions:
#     labels = result['labels']
#     scores = result['scores']
#     predicted_labels.append(labels)
#     predicted_scores.append(scores)
# # Create a DataFrame to store the results
# df_predictions = pd.DataFrame({
#     "comment_text": cleaned_place_text,
#     "predicted_label": predicted_labels,
#     "predicted_score": predicted_scores
# })

KeyboardInterrupt: 

In [ ]:
# # add column for the top label
# df_predictions['top_label'] = df_predictions['predicted_labels'].apply(lambda x: x[0])
# # add column for the top score
# df_predictions['top_score'] = df_predictions['predicted_scores'].apply(lambda x: x[0])

In [ ]:
# # save the predictions to a csv file
# df_predictions.to_csv('../outputs/five_percent_zero_shot_label.csv', index=False)